# Check-in 4 GPU runner (C2 + strength sweeps)

Runtime must be **GPU** (L4/A100). Then **Runtime → Run all**.

Installs MarkDiffusion, mounts Drive, stages Check-in 3 keys/images,
runs offline analysis/figures, then C2 imprint (SD 1.5) and crop/purification sweeps.


In [ ]:
# 0) Device check
import torch
print('cuda:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Switch runtime to GPU first (Runtime → Change runtime type)'


In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted')


In [ ]:
# 2) Clone / update repo
import pathlib, subprocess
REPO = pathlib.Path('/content/cmpe261-watermark-forgery')
if not (REPO / '.git').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/yashashav-dk/cmpe261-watermark-forgery.git', str(REPO)])
else:
    subprocess.check_call(['git', '-C', str(REPO), 'pull', '--ff-only'])
%cd /content/cmpe261-watermark-forgery
print('repo at', REPO)


In [ ]:
# 3) Install wmforge + pinned MarkDiffusion (same pin as Check-in 1/2/3)
import subprocess, sys
MARK = '9d81656d1a5f9e5194fc2f727bb795ef29e53809'
subprocess.check_call([
  sys.executable, '-m', 'pip', 'install', '-q',
  '-e', '.[attack]',
  f'markdiffusion @ git+https://github.com/THU-BPM/MarkDiffusion@{MARK}',
  'qrcode', 'galois', 'ldpc<2', 'matplotlib', 'pandas', 'markdown'
])
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'])
import torch, diffusers, markdiffusion
print('torch', torch.__version__, '| diffusers', diffusers.__version__, '| cuda', torch.cuda.is_available())
print('markdiffusion ok')


In [ ]:
# 4) Locate Check-in 3 full tree (keys + images) on Drive and stage it
from pathlib import Path
import shutil

candidates = list(Path('/content/drive').rglob('measurements-v2.json'))
print('found measurement logs:', len(candidates))
for p in candidates[:20]:
    print(' -', p)

checkin3 = None
for p in candidates:
    root = p.parent
    if (root / 'keys').exists() and (root / 'images').exists():
        checkin3 = root
        break
if checkin3 is None:
    for p in Path('/content/drive').rglob('checkin3'):
        full = p / 'full'
        if (full / 'keys').exists() and (full / 'images').exists() and (full / 'measurements-v2.json').exists():
            checkin3 = full
            break

assert checkin3 is not None, (
  'Could not find results/checkin3/full with keys/ and images/ on Drive. '
  'Upload that folder to Drive first.'
)
print('CHECKIN3 =', checkin3)
print('keys:', sorted(p.name for p in (checkin3 / 'keys').glob('*.pkl'))[:8])

dest = Path('results/checkin3/full')
dest.mkdir(parents=True, exist_ok=True)
for name in ['measurements-v2.json', 'manifest.json']:
    src = checkin3 / name
    if src.exists():
        shutil.copy2(src, dest / name)
for cal in checkin3.glob('*-calibration.json'):
    shutil.copy2(cal, dest / cal.name)
for sub in ['keys', 'images', 'inputs', 'checkpoints']:
    src = checkin3 / sub
    dst = dest / sub
    if src.exists():
        if dst.is_symlink() or dst.exists():
            if dst.is_symlink() or dst.is_file():
                dst.unlink()
            else:
                shutil.rmtree(dst)
        dst.symlink_to(src, target_is_directory=True)
        print('linked', sub, '->', src)
print('staged', dest)


In [ ]:
# 5) Offline Check-in 4 analysis + figures (CPU, fast)
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': 'src'}
subprocess.check_call([sys.executable, 'scripts/checkin4_analysis.py'], env=env)
subprocess.check_call([sys.executable, 'scripts/checkin4_figures.py'], env=env)
print('offline analysis/figures done')


In [ ]:
# 6) C2 imprint (SD 1.5 attacker) — long GPU job
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': 'src'}
cmd = [sys.executable, '-m', 'wmforge.checkin4', 'c2-imprint',
       '--checkin3', 'results/checkin3/full',
       '--out', 'results/checkin4/c2',
       '--device', 'cuda',
       '--schemes', 'GS', 'PRC', 'WIND', 'RI', 'SFW', 'TR']
print('Running:', ' '.join(cmd))
subprocess.check_call(cmd, env=env)
print('C2 imprint finished')


In [ ]:
# 7) Strength sweeps (crop + purification)
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': 'src'}
cmd = [sys.executable, '-m', 'wmforge.checkin4', 'sweep',
       '--checkin3', 'results/checkin3/full',
       '--out', 'results/checkin4/sweeps',
       '--device', 'cuda',
       '--schemes', 'GS', 'PRC', 'WIND', 'RI', 'SFW', 'TR',
       '--attacks', 'crop', 'purification']
print('Running:', ' '.join(cmd))
subprocess.check_call(cmd, env=env)
print('Sweeps finished')


In [ ]:
# 8) Zip results for download / Drive copy
import shutil, pathlib
out = pathlib.Path('/content/checkin4_gpu_results')
if out.exists():
    shutil.rmtree(out)
out.mkdir()
for p in ['results/checkin4/c2', 'results/checkin4/sweeps', 'results/checkin4/analysis', 'results/checkin4/figures']:
    src = pathlib.Path(p)
    if src.exists():
        shutil.copytree(src, out / src.name)
zip_path = shutil.make_archive('/content/checkin4_gpu_results', 'zip', out)
print('Download this zip:', zip_path)
drive_out = pathlib.Path('/content/drive/MyDrive/CMPE261_CheckIn4_GPU_Results')
drive_out.mkdir(parents=True, exist_ok=True)
shutil.copy2(zip_path, drive_out / 'checkin4_gpu_results.zip')
print('Copied to', drive_out)
